## Conditional Chains

In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import PydanticOutputParser,StrOutputParser
from langchain_core.runnables import RunnableLambda,RunnableBranch
from dotenv import load_dotenv
from pydantic import BaseModel, Field 
from typing import Literal

In [9]:
load_dotenv()

True

In [5]:
class Data(BaseModel):
    sentiment : Literal['Positive','Negative'] = Field(description='Generate the sentiment as either Positive or Nagitive')

In [6]:
parser = PydanticOutputParser(pydantic_object=Data)

In [ ]:
prompt_1 = PromptTemplate(template='Generate the sentiment of the following {feedback} in {format}', 
                          input_variables=['feedback'],partial_variables=({'format': parser.get_format_instructions()}))

In [10]:
model = ChatOpenAI(model='gpt-3.5-turbo')

In [11]:
chain = prompt_1 | model | parser

In [18]:
sentiments = chain.invoke({'feedback':'i didnt like the product'})

In [19]:
sentiments.sentiment

'Negative'

In [20]:
prompt_2 = PromptTemplate(template='Generate the Positive response for the received {sentiment} to the user', input_variables=['sentiment'])
prompt_3 = PromptTemplate(template='Generate the Response as the Sorry Message for the received {sentiment} to the user', input_variables=['sentiment'])

In [21]:
model_2 = ChatOpenAI(model='gpt-3.5-turbo')

In [22]:
strParser = StrOutputParser()

In [43]:
def default():
    return 'could not generate Sentiment'

In [44]:
conditional_chain = RunnableBranch((lambda x: x.sentiment == 'Positive', prompt_2 | model_2 | strParser),(lambda x: x.sentiment == 'Negative', prompt_3 | model_2 | strParser),RunnableLambda((lambda x : default)))

In [45]:
final_chain = chain | conditional_chain

In [48]:
result = final_chain.invoke({'I love my Wife'})

In [49]:
result

"That's great to hear! I'm glad you're feeling positive. Is there anything in particular that's making you feel this way?"

In [42]:
print(final_chain.get_graph().draw_ascii())

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOpenAI |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
